In [1]:
%pip install peft


  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/775.8 kB ? eta -:--:--
   --------------------------------------- 775.8/775.8 kB 16.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ------------------- -------------------- 5.8/11.6 MB 29.3 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 29.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 26.7 MB/s eta 0:00:00
Using cached shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
  Using cached https://download-r2.pytorch.org/whl/cu130/torch-2.13.0%2Bcu130-cp312-cp312-win_amd64.whl.metadata (39 kB)
  Using cached https://download-r2.pytorch.org/whl/cu130/torchvision-0.28.0%2Bcu130-cp312-cp312-win_amd64.whl.metadata (5.7 kB)
Using cached https://download-r2.pytorch.org/whl/cu130/torch-2.13.0%2Bcu130-cp312-cp312-win_amd64.whl (1915.5 MB)
Using cached https://download-r2.pytorch.org/whl/cu130/torchvision-0.28.0%2Bcu130-cp312-cp312-win_amd64.whl (9.1 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from diffusers import StableDiffusionInpaintPipeline, DDPMScheduler
import torch

model_id = "stable-diffusion-v1-5/stable-diffusion-inpainting"  # or stabilityai/stable-diffusion-2-inpainting

pipe = StableDiffusionInpaintPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
unet = pipe.unet
vae = pipe.vae
text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

vae.requires_grad_(False)
text_encoder.requires_grad_(False)  # freeze text encoder too — LoRA the UNet only, standard practice
vae.eval()
text_encoder.eval()

device = "cuda"
vae.to(device, dtype=torch.float16)
text_encoder.to(device, dtype=torch.float16)
unet.to(device, dtype=torch.float32)  # train UNet in fp32 for stability, or use fp16 + gradient scaling

d:\Stuff\PersonalProjects\augmented-gastric-histopathology-generator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Stuff\PersonalProjects\augmented-gastric-histopathology-generator\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Gawain\.cache\huggingface\hub\models--stable-diffusion-v1-5--stable-diffusion-inpainting. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows,

UNet2DConditionModel(
  (conv_in): Conv2d(9, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): CrossAttnDownBlock2D(
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True, bias=True)
          (proj_in): Conv2d(320, 320, kernel_size=(1, 1), stride=(1, 1))
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True, bias=True)
              (attn1): Attention(
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=320, out_features=320, bias=False)
                (to_v): Linear(i

In [2]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.05,
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

trainable params: 3,188,736 || all params: 862,724,100 || trainable%: 0.3696


In [7]:
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import os, json
from pathlib import Path

class GastricIMInpaintDataset(Dataset):
    def __init__(self, image_dir, mask_dir, tokenizer, size=512):
        # captions_json maps filename -> caption string, e.g.
        # "img001.png": "intestinal metaplasia, gastric antrum, incomplete type"
        # with open(captions_json) as f:
        #     self.captions = json.load(f)
        self.image_dir = image_dir
        directory = Path(image_dir)
        self.filenames = [f.name for f in directory.iterdir() if f.is_file()]
        self.mask_dir = mask_dir
        # self.filenames = list(self.captions.keys())
        self.captions = {fname: "intestinal metaplasia, gastric mucosa" for fname in self.filenames}
        self.tokenizer = tokenizer
        self.image_transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image = Image.open(os.path.join(self.image_dir, fname)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, fname)).convert("L")  # gland mask, white = region to inpaint

        image_t = self.image_transform(image)
        mask_t = self.mask_transform(mask)
        mask_t = (mask_t > 0.5).float()

        masked_image_t = image_t * (1 - mask_t)  # black out masked region

        caption = self.captions[fname]
        input_ids = self.tokenizer(
            caption, padding="max_length", truncation=True,
            max_length=self.tokenizer.model_max_length, return_tensors="pt"
        ).input_ids[0]

        return {
            "pixel_values": image_t,
            "mask": mask_t,
            "masked_image": masked_image_t,
            "input_ids": input_ids,
        }

dataset = GastricIMInpaintDataset(
    "../datasets/healthy2im/trainB",
    "../datasets/healthy2im/masks",
    tokenizer,
)

In [8]:
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=4)

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=1e-4, weight_decay=1e-6
)

from diffusers.optimization import get_cosine_schedule_with_warmup
num_epochs = 15
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=100,
    num_training_steps=len(dataloader) * num_epochs,
)

scaling_factor = vae.config.scaling_factor

for epoch in range(num_epochs):
    for step, batch in enumerate(dataloader):
        images = batch["pixel_values"].to(device, dtype=torch.float16)
        masks = batch["mask"].to(device, dtype=torch.float16)
        masked_images = batch["masked_image"].to(device, dtype=torch.float16)
        input_ids = batch["input_ids"].to(device)

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample() * scaling_factor
            masked_latents = vae.encode(masked_images).latent_dist.sample() * scaling_factor
            mask_latent = torch.nn.functional.interpolate(masks, size=latents.shape[-2:])
            encoder_hidden_states = text_encoder(input_ids)[0]

        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # SD Inpainting's UNet expects 9 channels: noisy_latents(4) + mask(1) + masked_latents(4)
        unet_input = torch.cat([noisy_latents, mask_latent, masked_latents], dim=1).to(torch.float32)

        noise_pred = unet(
            unet_input, timesteps,
            encoder_hidden_states=encoder_hidden_states.to(torch.float32)
        ).sample

        # supervise loss primarily inside the masked region
        loss = torch.nn.functional.mse_loss(
            noise_pred.float() * mask_latent, noise.float() * mask_latent
        )
        # optionally add a small full-image loss term to keep boundary coherence:
        # loss += 0.1 * F.mse_loss(noise_pred.float(), noise.float())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        if step % 50 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f}")

    unet.save_pretrained(f"../checkpoints/lora_gastric_im_inpaint/epoch_{epoch}/")

In [ ]:
from peft import get_peft_model_state_dict
lora_state_dict = get_peft_model_state_dict(unet)
torch.save(lora_state_dict, "../checkpoints/lora_gastric_im_inpaint_final.pt")

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image

pipe = StableDiffusionInpaintPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to("cuda")
pipe.unet = unet  # your LoRA-adapted unet (merge_and_unload() first if you want a plain UNet object)

image = Image.open("healthy_gastric_patch.png").convert("RGB")
mask = Image.open("healthy_gland_mask.png").convert("L")  # white = region to convert to IM

result = pipe(
    prompt="intestinal metaplasia, gastric antrum, goblet cells, incomplete type",
    image=image,
    mask_image=mask,
    strength=0.99,        # near-1.0 = fully regenerate masked region from noise
    guidance_scale=7.5,
    num_inference_steps=50,
).images[0]

result.save("edited_gastric_patch.png")